# Project 02 (medium): rescuing a dirty data set

**Goal:** a realistically dirty order data set (500 orders of an online shop) is to
be made analysable. You find the problems **yourself**, treat them **with
justification**, and prove at the end with an acceptance test that everything is clean.

**Preparation** (once, in the folder `02-medium`, venv active):

```
python generate_data.py
```

The data are **synthetic** (why is explained at the top of `generate_data.py`) —
but every built-in problem occurs constantly in real data.
**No cheating:** search yourself first, then look into the generator if necessary.

**Reference to the script:** sections 2.1 (cleaning), 2.3 (groupby), 1.4 (box plot).

**Working rule for this project:** for every cleaning decision you write one sentence
of justification into the markdown cells provided ("Decision: ..."). This is not
harassment — undocumented cleaning is not reproducible (script 3.3).

## 1. Inspect: what is wrong here?

**Task:** get an overview with `head()`, `info()`, `describe(include="all")` and some
samples, and note in the markdown cell below a list of all the problems you find.
(There are at least 6.)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PATH = "datasets/orders_raw.csv"
if not os.path.exists(PATH):
    PATH = "../" + PATH

raw = pd.read_csv(PATH)
print(raw.shape)
raw.head(8)

In [ ]:
raw.info()
raw.describe(include="all").T

**Your list of problems** *(fill in here before reading on!)*:

1. ...
2. ...

<details><summary>For comparison: the complete list (unfold only after your own search)</summary>

1. `price` is text ("49,99 EUR") instead of a number — comma and unit
2. missing prices (empty fields → NaN)
3. extreme price outliers (four digits and more — looks like a decimal-point error)
4. `customer_age` contains the special code −999 (and at least one impossible value)
5. `city` inconsistent: capitalisation, spaces, spelling variants
6. `date` mixed in two formats (ISO and day-first)
7. duplicate rows (515 rows, but only 500 distinct `order_id`s?)
</details>

## 2. Duplicates

**Task:** how many exact duplicates are there? Remove them (result: `df`).
Then check whether `order_id` is unique too (otherwise there would be near-duplicates).

In [ ]:
# TODO: raw.duplicated().sum(), drop_duplicates(), df["order_id"].is_unique
print("rows:", len(df))   # expected: 500

**Decision:** exact duplicates removed (15 of them) — identical rows including the
order id cannot be genuine repeat orders, they are an export error.

## 3. Price: text to number, missing values

**Task:** create a numeric column `price_eur` (remove the unit, comma to point,
`astype(float)` — empty fields should automatically become `NaN`).
How many prices are missing?

In [ ]:
# TODO: df["price_eur"] = ... (str.replace twice, then astype(float))
print("missing prices:", df["price_eur"].isna().sum())   # expected: 12

**Decision:** the 12 missing prices stay `NaN` (no imputation): they are only 2.4 %
of the rows, pandas aggregations ignore NaN automatically, and filling an order price
with the median would feign a precision we do not have.

## 4. Outliers — the favourite trap

**Task (a):** apply the IQR rule (script 2.1) to `price_eur` **globally**.
How many rows are flagged? Look at the flagged prices — are they all errors?

In [ ]:
# TODO: q1, q3 = df["price_eur"].quantile([0.25, 0.75]); iqr; upper bound; mask; print count + values

**41 flagged rows — but only 3 really look broken** (four digits and more).
The rest are expensive but plausible electronics prices. The global IQR rule compares
books with televisions — unfair. The box plot per category shows the problem:

In [ ]:
import seaborn as sns
sns.boxplot(data=df, x="category", y="price_eur")
plt.yscale("log")   # log scale, otherwise the outliers squash everything together
plt.title("Prices per category (log scale) — the 3 genuine outliers stand out")
plt.show()

**Task (b):** apply the IQR rule **per category**
(hint: `df.groupby("category")["price_eur"].transform(...)` with a function that
returns the upper bound per group). Now exactly **3** rows should be flagged.
Correct them: they are obviously decimal-point errors (factor 100) → divide by 100.

In [ ]:
# TODO: function upper_iqr_bound(g); transform; mask; show flagged rows; correct with /100
print("new maximum price:", df["price_eur"].max().round(2))

**Decision:** 3 prices (1321 / 18363 / 19685 EUR) classified as decimal-point errors
and divided by 100 — the justification: they lie exactly two orders of magnitude above
what is usual for their category, and after the correction they fit inconspicuously
into the price band. *(In real life you would additionally check the source system.)*
Note the triad from script 2.1: error → correct; genuine extreme value → keep;
special code → NaN. Here we had errors — the expensive televisions above, by contrast,
were genuine extreme values and stay untouched!

## 5. Normalising the cities

**Task:** `df["city"].value_counts()` shows the chaos. Create `city_clean`:
strip spaces, lowercase, unify the spelling variants
(`münchen`→`munich`, `köln`→`cologne`), then `str.capitalize()`.
At the end: exactly **5** different cities.

In [ ]:
print(df["city"].value_counts())
# TODO: df["city_clean"] = ... (strip, lower, replace mapping, capitalize)
print("\nafterwards:", sorted(df["city_clean"].unique()))
print("count:", df["city_clean"].nunique())   # expected: 5

## 6. Age: special code and plausibility

**Task:** `df["customer_age"].describe()` reveals two problems (look at min and max!).
Create `age`: replace the special code **−999** ("no answer") with `NaN` and also set
implausible ages (over 100) to `NaN`.

In [ ]:
print(df["customer_age"].describe().round(1))
# TODO: df["age"] = ... (replace(-999, np.nan); then age > 100 to np.nan)
print("\nmissing:", df["age"].isna().sum(), "| range:", df["age"].min(), "-", df["age"].max())

**Decision:** −999 is obviously a "no answer" code (25 times) → NaN.
The age 234 is physically impossible and the true value cannot be reconstructed → NaN
rather than guessing. In total 26 missing ages (5.2 %) — for analyses by age these rows
are left out automatically, the rest of the data set stays usable.

## 7. Date: two formats, one column

**Task:** `pd.to_datetime` with a fixed `format` and `errors="coerce"` parses only the
matching format (the rest becomes NaT). Parse both formats separately and combine them
with `fillna`. At the end: **0** NaT values.

In [ ]:
# TODO: parse both formats separately (errors="coerce"), combine with fillna
print("not parseable:", df["date_clean"].isna().sum())   # expected: 0
print(df["date_clean"].min(), "to", df["date_clean"].max())

## 8. Acceptance test

This cell is given as is. If it runs through without an error, your data set is
officially clean.

In [ ]:
clean = df[["order_id", "date_clean", "city_clean", "category",
            "price_eur", "quantity", "age"]].rename(
                columns={"date_clean": "date", "city_clean": "city"})

assert len(clean) == 500, "duplicates not (properly) removed"
assert clean["order_id"].is_unique
assert clean["price_eur"].dtype == float and clean["price_eur"].isna().sum() == 12
assert clean["price_eur"].max() < 500, "price outliers not corrected"
assert clean["city"].nunique() == 5, "cities not fully normalised"
assert clean["age"].isna().sum() == 26 and clean["age"].max() <= 100
assert str(clean["date"].dtype).startswith("datetime64") and clean["date"].isna().sum() == 0
assert abs(clean["price_eur"].median() - 71.6) < 0.5, "prices are wrong (check the correction)"

print("ACCEPTANCE PASSED — the data set is clean.")
clean.head()

## 9. The reward: the first real analysis

Now that the data are clean, the analysis is a three-liner — given as is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
clean.groupby("category")["price_eur"].mean().sort_values().plot.barh(ax=axes[0])
axes[0].set_title("Mean price per category (EUR)")
clean.set_index("date").resample("ME")["price_eur"].sum().plot(ax=axes[1])
axes[1].set_title("Revenue per month (EUR)")
plt.tight_layout(); plt.show()

clean.groupby("city").agg(orders=("order_id", "count"),
                          revenue=("price_eur", "sum"),
                          mean_age=("age", "mean")).round(1)

## Done — what you can do now

- *find* data problems systematically instead of only fixing them
- type conversion, duplicates, special codes, text normalisation, mixed date formats
- use the IQR rule correctly — including the lesson that it has to be applied
  **per group** when the groups live on different scales
- document cleaning decisions and secure them with an acceptance test

**Bonus tasks** (optional):
1. Open `generate_data.py` and compare: did you find all the built-in problems?
2. Change the seed in the generator — does your notebook still run through? (The
   acceptance numbers change; which checks can be formulated independently of the seed?)
3. Build a function `clean_data(df_raw)` that encapsulates all the steps — the first
   step from an analysis to a reusable pipeline.